# ED Journal

> Processing journal events

In [ ]:
#| default_exp edjournal

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import time
import glob
import json
import logging
import datetime
import ntpath
from typing import Iterator
from functools import reduce


In [ ]:
from confproxy.core import init_console_logging
from edcompanion.core import configuration


In [ ]:
configuration["FOLDERS"]["ed_journal_archive"]

'C:\\Users\\fenke\\Saved Games\\Elite Dangerous'

In [ ]:
init_console_logging(__name__)

2025-12-23T00:39:18+0100 INFO	30688	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| export
syslog = logging.getLogger(__name__)


In [ ]:
? os.listdir

Signature:  os.listdir(path=None)
Docstring:
Return a list containing the names of the files in the directory.

path can be specified as either str, bytes, or a path-like object.  If path is bytes,
  the filenames returned will also be bytes; in all other circumstances
  the filenames returned will be str.
If path is None, uses the path='.'.
On some platforms, path may also be specified as an open file descriptor;\
  the file descriptor must refer to a directory.
  If this functionality is unavailable, using it raises NotImplementedError.

The list is in arbitrary order.  It does not include the special
entries '.' and '..' even if they are present in the directory.
Type:      builtin_function_or_method

In [ ]:
#| export

def list_journals_unsorted(journalpath):
    for jn in (os.path.join(journalpath, f) for f in os.listdir(journalpath) if 'Journal' in f.split('.')[0] and '.log' in f):
        yield jn


In [ ]:
print(list(list_journals_unsorted(configuration["FOLDERS"]["ed_journals"])))

['C:\\Users\\fenke\\Saved Games\\Frontier Developments\\Elite Dangerous\\Journal.2025-12-18T121712.01.log']


In [ ]:
#| export

def list_journals_sorted(journalpath):
    return sorted(
            [os.path.join(journalpath, f) for f in os.listdir(journalpath) if 'Journal' in f.split('.')[0] and '.log' in f],
            key=lambda f:f.replace('-', '').replace('Journal.20', 'Journal.').replace('T','')
        )


In [ ]:
#| export

def list_journals(journalpath, sorted=True):
    if sorted:
        for jn in list_journals_sorted(journalpath):
            yield jn
    else:
        for jn in list_journals_unsorted(journalpath):
            yield jn

In [ ]:
list(list_journals(configuration["FOLDERS"]["ed_journal_archive"]))

['C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310101330.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310105112.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310202304.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310214711.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310222232.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310223026.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220310223213.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220312115912.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220312130140.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220313111717.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220313152207.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.220314163549.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite D

In [ ]:
#| export

def read_journal(
    journal:str,    # path to journal file
    tail=True       # finish on end of input or wait for new events
)->Iterator[dict]:
    """
        Returns a generator of journal events
        journal: path to journal
        notail:  finish on end of input or wait for new events (finishes on 'Shutdown' event)
    """

    syslog = logging.getLogger(f"root.{__name__}")
    last_timestamp = None
    
    try:
        syslog.info(f"\nReading journal: {ntpath.basename(journal)}")

        with open(journal, encoding="utf-8") as journalfile:
            while True: # not shutdown_seen:
                line = journalfile.readline()
                if not line:
                    if tail:
                        time.sleep(0.3)
                        continue
                    else:
                        break

                if len(line) < 5:
                    continue

                try:
                    event = json.loads(line)
                    if not last_timestamp:
                        event['logfile'] = ntpath.basename(journal)
                        
                    last_timestamp = event.get('timestamp', last_timestamp)

                except json.decoder.JSONDecodeError as JX:
                    syslog.exception("Exception: %s", JX, exc_info=True, stack_info=True)
                    return

                yield event
        
                if event.get('event', '') == 'Shutdown':
                    syslog.info(f"SHUTDOWN {event.get('timestamp'):22} {ntpath.basename(journal)}")
                    break



    except KeyboardInterrupt:
        syslog.info("Keyboard Interrupt")
        yield dict(
            event='KeyboardInterrupt',
            timestamp=last_timestamp,
            filename=f"{ntpath.basename(journal)}"
        )

    finally:
        syslog.info(f"Done reading journal: {journal}")
        yield dict(
            event='JournalFinished',
            timestamp=last_timestamp,
            filename=f"{ntpath.basename(journal)}"
        )



In [ ]:
backlog = 8


logfiles = list_journals_sorted(configuration["FOLDERS"]["ed_journal_archive"])

backlog = min(backlog, len(logfiles)-1)
print(backlog)
print(json.dumps(logfiles[-(1+backlog):], indent=4))

8
[
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-09-25T233500.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-09-30T151545.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T115609.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T155546.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T175708.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T182622.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-15T205848.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-16T201712.01.log",
    "C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-11-07T143606.01.log"
]


In [ ]:
backlog = '2024-09-30'
reduce(
                lambda t, j: t if not t and backlog not in j else t + [j],
                logfiles, []
            )

['C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-09-30T151545.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T115609.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T155546.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T175708.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-10T182622.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-15T205848.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-10-16T201712.01.log',
 'C:\\Users\\fenke\\Saved Games\\Elite Dangerous\\Journal.2024-11-07T143606.01.log']

In [ ]:
#| export

def track_journals(journalpath, backlog=0):
    '''Iterable for Journal events spanning multiple log files'''

    try:

        #logfiles = sorted(glob.glob(os.path.join(journalpath, journalglob)))
        logfiles = list_journals_sorted(journalpath)

        if isinstance(backlog, int):
            backlog = min(backlog, len(logfiles)-1)
            syslog.info(f"Reading journals, backlog = {backlog}")
            for f in logfiles[-(1+backlog):]:
                yield from read_journal(f)

        elif isinstance(backlog, str):
            syslog.info(f"Reading journals, backlog = {backlog}")
            for f in reduce(
                lambda t, j: t if not t and backlog not in j else t + [j],
                logfiles, []
            ):
                yield from read_journal(f)


    except KeyboardInterrupt as kbi:
        syslog.info(f"Keyboard Interrupt {kbi.info()}")
        pass



In [ ]:
for j in track_journals(configuration["FOLDERS"]["ed_journal_archive"], backlog='2024-09-30'):
    print(json.dumps(j, indent=4))

2025-12-23T00:39:28+0100 INFO	30688	__main__	433723690.py	track_journals	18	Reading journals, backlog = 2024-09-30


{
    "timestamp": "2024-09-30T13:15:39Z",
    "event": "Fileheader",
    "part": 1,
    "language": "English/UK",
    "Odyssey": true,
    "gameversion": "4.0.0.1809",
    "build": "r305601/r0 ",
    "logfile": "Journal.2024-09-30T151545.01.log"
}
{
    "timestamp": "2024-09-30T13:16:13Z",
    "event": "Commander",
    "FID": "F9569960",
    "Name": "immerlicht"
}
{
    "timestamp": "2024-09-30T13:16:13Z",
    "event": "Materials",
    "Raw": [
        {
            "Name": "zinc",
            "Count": 134
        },
        {
            "Name": "sulphur",
            "Count": 110
        },
        {
            "Name": "niobium",
            "Count": 14
        },
        {
            "Name": "phosphorus",
            "Count": 113
        },
        {
            "Name": "iron",
            "Count": 113
        },
        {
            "Name": "carbon",
            "Count": 113
        },
        {
            "Name": "vanadium",
            "Count": 3
        },
        {
       

In [ ]:
event_types = set()
for j in track_journals(configuration["FOLDERS"]["ed_journal_archive"], backlog=10):
    event_types.add(j['event'])


2025-12-23T00:39:37+0100 INFO	30688	__main__	433723690.py	track_journals	13	Reading journals, backlog = 10


In [ ]:
import nbdev; nbdev.nbdev_export()

c:\Users\fenke\repos\EDTravelCompanion\.venv\Lib\site-packages\nbdev\export.py:88: UserWarning: Notebook 'c:\Users\fenke\repos\EDTravelCompanion\nbs\eddb\82_edsm.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"
